In [ ]:
import json
import logging
import os
import subprocess
import sys
import warnings
from pathlib import Path

os.environ.setdefault("LOGURU_LEVEL", "INFO")
os.environ.setdefault("NEMO_LOG_LEVEL", "ERROR")
warnings.filterwarnings("ignore", message=r".*concurrency.*deprecated.*")
logging.getLogger("nemo_logger").setLevel(logging.ERROR)

# Hindi Indic ASR Benchmark Tutorial

This notebook follows the paired-backend methodology used by the ALM tutorial while executing the canonical `audio_indic_asr_benchmark.py` implementation. Both runs use the same pinned 531.7738-hour Hindi cohort, model engines, processor graph, concurrency, and validators; only the executor changes.

The benchmark-equivalent environment is one 128-CPU, 8× H100 node with compatible TensorRT and TensorRT-LLM runtimes.

## Step 1: Prepare the pinned public cohort

From the repository root, run:

```bash
python benchmarking/data_prep/prepare_audio_indic_asr_data.py \
  --output-path /data/audio_indic_asr_parakeet_hindi_531h_35376a11 \
  --cache-dir /data/_hf_cache/audio_indic_asr
```

This pins `ketav/parakeet-hindi-asr` at revision `35376a112c4b79318eeaba0c0dd1b6f1a9bf0ea0` and validates all 216,169 clips.

In [ ]:
# Run this notebook from the Curator repository root and update the three paths.
REPO_ROOT = Path.cwd().resolve()
INPUT_MANIFEST = Path("/data/audio_indic_asr_parakeet_hindi_531h_35376a11/manifest.jsonl")
INDIC_CANARY_ENGINE_DIR = Path("/models/audio_indic_asr/indic_canary_trtllm/engine_bfloat16_64_new")
PARAKEET_TENSORRT_ENGINE_DIR = Path("/models/audio_indic_asr/parakeet_indic_trt/encoder_fp16_b1_o8_m16_f8_o800_m4001")
OUTPUT_ROOT = REPO_ROOT / "indic_asr_benchmark_output"

required_paths = [INPUT_MANIFEST, INDIC_CANARY_ENGINE_DIR, PARAKEET_TENSORRT_ENGINE_DIR]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    msg = f"Update the tutorial paths; missing: {missing}"
    raise FileNotFoundError(msg)

## Step 2: Define the exact runner

The Hydra tutorial runner delegates to the canonical benchmark function, including its input normalization, four-way sharding, model/runtime preflight, full processor chain, timer, and 1:1 output validation. Each backend gets a fresh result directory.

In [ ]:
def run_backend(backend: str) -> dict:
    output_dir = OUTPUT_ROOT / backend
    if output_dir.exists():
        msg = f"Use a fresh output path: {output_dir}"
        raise FileExistsError(msg)
    command = [
        sys.executable,
        str(REPO_ROOT / "tutorials/audio/indic_asr/main.py"),
        "--config-path",
        ".",
        "--config-name",
        "pipeline",
        f"input_manifest={INPUT_MANIFEST}",
        f"indic_canary_engine_dir={INDIC_CANARY_ENGINE_DIR}",
        f"parakeet_tensorrt_engine_dir={PARAKEET_TENSORRT_ENGINE_DIR}",
        f"benchmark_results_path={output_dir}",
        f"backend={backend}",
    ]
    subprocess.run(command, cwd=REPO_ROOT, check=True)  # noqa: S603
    return json.loads((output_dir / "metrics.json").read_text())

## Step 3: Run Xenna

On the standard EOS 8× H100 runner, the accepted full-process wall-time range is 600–900 seconds.

In [ ]:
xenna_metrics = run_backend("xenna")
xenna_metrics

## Step 4: Run Ray Data

Ray Data is an ungated same-data comparison. Wait for the Xenna process and its Ray cluster to stop before starting it.

In [ ]:
ray_data_metrics = run_backend("ray_data")
ray_data_metrics

## Step 5: Compare the paired evidence

Cardinality, duration, prediction coverage, provenance, and WER coverage should satisfy the same correctness contract for both executors. Only Xenna has the 10–15 minute EOS timing criterion.

In [ ]:
keys = [
    "time_taken_s",
    "pipeline_time_s",
    "num_output_rows",
    "total_audio_duration_hours",
    "input_output_coverage_ratio",
    "best_prediction_coverage_ratio",
    "wer_output_coverage_ratio",
    "mean_wer_pct",
]
comparison = {key: {"xenna": xenna_metrics[key], "ray_data": ray_data_metrics[key]} for key in keys}
comparison